In [1]:
from transformers import VisionEncoderDecoderConfig

# Replace 'your-model-name' with your model repository name or local path
config = VisionEncoderDecoderConfig.from_pretrained("AfriMM/SiglipNllb")

/opt/conda/envs/afrimmd/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ValueError: A configuraton of type vision-encoder-decoder cannot be instantiated because not both `encoder` and `decoder` sub-configurations are passed, but only {'attn_implementation': None}

In [ ]:
from huggingface_hub import hf_hub_download
# import torch


repo_id = "AfriMM/SiglipNllb"  # Replace with the actual repo id
filename = "model.pth"

# Download the file; this returns the local path where the file is stored.
model_path = hf_hub_download(repo_id=repo_id, filename=filename)



In [3]:
model_path

'/home/mardiyyahodu/.cache/huggingface/hub/models--AfriMM--SiglipNllb/snapshots/f62b9cba866ed4d3fcc1706b10d00f2ea3cf8ec5/model.pth'

In [ ]:
'/home/mardiyyahodu/.cache/huggingface/hub/models--AfriMM--SiglipNllb/snapshots/f62b9cba866ed4d3fcc1706b10d00f2ea3cf8ec5/model.pth'

In [5]:
from transformers import (
    AutoProcessor,
    AutoTokenizer,
    SiglipVisionModel,
    M2M100ForConditionalGeneration,
)
from model import VisionEncoderDecoderModel
import torch

In [12]:
device = "cuda"

In [ ]:

vision_encoder = SiglipVisionModel.from_pretrained("google/siglip-base-patch16-256-multilingual")
decoder = M2M100ForConditionalGeneration.from_pretrained("facebook/nllb-200-distilled-600M").model.decoder
model = VisionEncoderDecoderModel(encoder=vision_encoder, decoder=decoder)

model.config.bos_token_id = decoder.config.bos_token_id
model.config.eos_token_id = decoder.config.eos_token_id
model.config.pad_token_id = decoder.config.pad_token_id

checkpoint = torch.load('/home/mardiyyahodu/.cache/huggingface/hub/models--AfriMM--SiglipNllb/snapshots/f62b9cba866ed4d3fcc1706b10d00f2ea3cf8ec5/model.pth')
fixed_state_dict = {k.replace('_orig_mod.', ''): v for k, v in checkpoint['model_state_dict'].items()}
model.load_state_dict(fixed_state_dict)
model.to(device)
model = torch.compile(model)

/var/tmp/ipykernel_12249/3001415633.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('/home/mardiyyahodu/.cache/huggingface/hub/models--AfriMM--S